In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Copy folder from Drive to local Colab space
!cp -r /content/drive/MyDrive/dataset


cp: missing destination file operand after '/content/drive/MyDrive/dataset'
Try 'cp --help' for more information.


In [ ]:
import os
import shutil

src_path = '/content/drive/MyDrive/dataset'
dst_path = '/content/dataset'

# Check if the path exists
if os.path.exists(src_path):
    print(f"Found 'dataset' at: {src_path}")
    print(f"Is it a directory? {os.path.isdir(src_path)}")

    # Copy using Python shutil
    if os.path.exists(dst_path):
        shutil.rmtree(dst_path) # Clean up if it already exists

    shutil.copytree(src_path, dst_path)
    print("Successfully copied dataset to Colab local storage!")
else:
    print(f"Error: {src_path} could not be resolved. Please try running the mount command again.")


Found 'dataset' at: /content/drive/MyDrive/dataset
Is it a directory? True
Successfully copied dataset to Colab local storage!


In [ ]:
!pip install ultralytics


In [ ]:
yaml_path = '/content/dataset/dataset.yaml'

yaml_content = """path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: standing
  1: falling
  2: Fall
"""

with open(yaml_path, 'w') as file:
    file.write(yaml_content)

print("Successfully rewrote dataset.yaml for Colab. Current contents:")
print(open(yaml_path).read())


Successfully rewrote dataset.yaml for Colab. Current contents:
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: standing
  1: falling
  2: Fall



In [ ]:
from ultralytics import YOLO

# Load the pretrained segmentation model
model = YOLO('yolov8n-seg.pt')

# Train using GPU (device=0)
results = model.train(
    data='/content/dataset/dataset.yaml',
    epochs=100,
    imgsz=320,  # Match original frame resolution (320x240) to speed up training
    device=0
)


Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

In [ ]:
import glob

# Search for all trained weights (.pt files) in the runs directory
pt_files = glob.glob('/content/runs/**/*.pt', recursive=True)

if pt_files:
    print("Found trained weight files:")
    for path in pt_files:
        print(f" - {path}")
else:
    print("No .pt weight files found. Please ensure training completed successfully.")



Found trained weight files:
 - /content/runs/segment/train-2/weights/last.pt
 - /content/runs/segment/train-2/weights/best.pt


In [ ]:
!cp YOUR_FOUND_PATH /content/drive/MyDrive/yolov8_fall_detection_best.pt


cp: cannot stat 'YOUR_FOUND_PATH': No such file or directory


In [ ]:
!cp /content/runs/segment/train-2/weights/best.pt /content/drive/MyDrive/yolov8_fall_detection_best.pt

In [16]:
from ultralytics import YOLO

# 1. Load your best trained weights
model = YOLO('/content/drive/MyDrive/yolov8_fall_detection_best.pt')  # or local path on PC

# 2. Run validation on the dataset
metrics = model.val(data='/content/dataset/dataset.yaml')  # or local dataset.yaml path

# 3. Print Bounding Box and Segmentation metrics
print("\n=== Bounding Box (Detection) Metrics ===")
print(f"Precision (Accuracy of detections): {metrics.results_dict['metrics/precision(B)']:.3f}")
print(f"Recall (Percent of falls detected):  {metrics.results_dict['metrics/recall(B)']:.3f}")
print(f"mAP50 (General detection score):      {metrics.results_dict['metrics/mAP50(B)']:.3f}")

print("\n=== Polygon Mask (Segmentation) Metrics ===")
print(f"Precision (Accuracy of shape masks): {metrics.results_dict['metrics/precision(M)']:.3f}")
print(f"Recall (Percent of shapes matched):  {metrics.results_dict['metrics/recall(M)']:.3f}")
print(f"mAP50 (General segmentation score):   {metrics.results_dict['metrics/mAP50(M)']:.3f}")


Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,649 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 657.6±300.1 MB/s, size: 24.9 KB)
val: Scanning /content/dataset/labels/val.cache... 109 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 109/109 106.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.9it/s 3.6s
                   all        109        109      0.996      0.996      0.995      0.916      0.996      0.996      0.995      0.867
              standing         57         57      0.991          1      0.995      0.933      0.991          1      0.995      0.835
               falling          6          6          1      0.989      0.995      0.858          1      0.989      0.995      0.895
                  Fall         46         46    